# Using Predictive Analytics to Identify Engagement Drivers in Instagram Reels
## Notebook 2 of 4: Data Preparation
**CRISP-DM phase 3**  |  Previous: `01_business_and_data_understanding`  |  Next: `03_modeling`

**Input:** `../Data/raw_data.csv`  |  **Output:** `../Data/cleaned_data.csv`

The CRISP-DM data preparation tasks are carried out in order: **select** the data, **clean** it, **construct** new features, and **format** it for modeling. Cleaning happens only in this notebook; Notebook 1 records what needs fixing. Between constructing the derived features and the interaction features, section 3.4 explores the cleaned data (the second pass of Data Understanding).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option('display.max_columns', None)

reels_df = pd.read_csv('../Data/raw_data.csv')
print(f"Raw data: {reels_df.shape[0]} rows, {reels_df.shape[1]} columns")

## 3.1 Select data

### Roles of the variables
| Role | Variables | Used as predictors? |
|---|---|---|
| **Target** | `performance_bucket` | n/a |
| **Content framing and structure** | `hook_category`, `reel_structure`, `craft_category` | Yes |
| **Delivery mechanics** | `result_shown_first`, `voice_present`, `audio_not_required`, `pov_present` | Yes |
| **Context** (kept in the dataset, not used as predictors) | `creator_handle`, `total_followers`, `date_posted`, `collection_date`, `days_since_posted` | No. The target is already creator-relative, which accounts for audience size, and Reel age is treated as a confounder (see section 3.4) |
| **Outcome measures** (excluded to avoid leakage) | `views`, `likes`, `comments`, `engagement_rate`, `like_rate`, `comment_rate`, `views_per_follower` | No. They are components of the engagement outcome, so using them to predict it would be circular |
| **Identifier** | `reel_id` (extracted from `url`) | No |
| **Dropped** | `url`, `hook_text` | No. `url` is only needed to extract the Reel ID, and `hook_text` is free text that is missing when the hook is visual |

### Create a unique Reel ID

In [ ]:
# The Reel ID is the segment after /reel/, /reels/, or /p/ in the URL,
# e.g. https://www.instagram.com/reel/DT5we1ZAQS4/
reels_df['reel_id'] = reels_df['url'].str.extract(r'instagram\.com/(?:reel|reels|p)/([^/?#]+)', expand=False)

# A URL with no Reel ID (for example a profile link) keeps its full URL as the identifier
no_id = reels_df['reel_id'].isna()
print("URLs without a Reel ID (full URL kept as the identifier):", no_id.sum())
reels_df.loc[no_id, 'reel_id'] = reels_df.loc[no_id, 'url']

### Resolve duplicate Reels
Notebook 1 found Reels that appear twice with identical content but **opposite performance labels**. A Reel cannot be both baseline and high-performing, and the copies would also land in both the training and validation sets, so they must be resolved before modeling. Reels that are exact duplicates (same label) keep a single copy.

Which label is correct depends on how the data was collected, so the rule is a setting:
- `'drop_all'` (default) removes every copy of a conflicting Reel, because the label cannot be trusted.
- `'keep_high'` keeps the copy labeled high-performing (for example, if baseline Reels were sampled from recent posts that may also be top posts).

In [ ]:
DUPLICATE_RULE = 'drop_all'   # or 'keep_high'

rows_before = len(reels_df)

# 1) exact duplicates (same URL and same label): keep one copy
reels_df = reels_df.drop_duplicates(subset=['url', 'performance_bucket'])

# 2) URLs that still appear more than once have conflicting labels
conflict = reels_df['url'].duplicated(keep=False)
print(f"Reels with conflicting labels: {reels_df.loc[conflict, 'url'].nunique()} ({conflict.sum()} rows)")

if DUPLICATE_RULE == 'drop_all':
    reels_df = reels_df[~conflict]
elif DUPLICATE_RULE == 'keep_high':
    is_high = reels_df['performance_bucket'].astype(str).str.strip().str.lower().eq('high')
    reels_df = reels_df.assign(_is_high=is_high).sort_values('_is_high', ascending=False)
    reels_df = reels_df.drop_duplicates(subset='url').sort_index().drop(columns='_is_high')
else:
    raise ValueError("DUPLICATE_RULE must be 'drop_all' or 'keep_high'")

print(f"Rows: {rows_before} -> {len(reels_df)}")
assert not reels_df['url'].duplicated().any()

# drop columns that are not features, and put the identifier first
reels_df = reels_df.drop(columns=['url', 'hook_text'])
reels_df = reels_df[['reel_id'] + [c for c in reels_df.columns if c != 'reel_id']].reset_index(drop=True)

## 3.2 Clean data

### Rename columns

In [ ]:
reels_df = reels_df.rename(columns={
    'views_raw': 'views',
    'likes_raw': 'likes',
    'comments_raw': 'comments'
})

### Standardize text and encode Y/N values
Category labels are converted to lowercase, trimmed, and joined with underscores (for example `Fine Art (Paint)` becomes `fine_art_paint`). The four delivery variables are encoded as 1 (`Y`) and 0 (`N`).

In [ ]:
categorical_cols = ['craft_category', 'reel_structure', 'hook_category']
binary_cols = ['pov_present', 'result_shown_first', 'voice_present', 'audio_not_required']

for col in categorical_cols:
    reels_df[col] = (
        reels_df[col].astype(str).str.lower().str.strip()
        .str.replace(' / ', '_', regex=False)
        .str.replace(' ', '_', regex=False)
        .str.replace('[()]', '', regex=True)
    )

for col in binary_cols:
    reels_df[col] = (
        reels_df[col].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0})
    )

assert reels_df[binary_cols].notna().all().all(), "Unexpected value in a Y/N column"

### Convert counts to numbers
`total_followers`, `views`, `likes`, and `comments` are stored as text such as `953K` or `1.2M`.

In [ ]:
def convert_to_number(x):
    """Convert strings such as '113K' or '1.2M' to numbers."""
    x = str(x).strip().upper().replace(',', '')
    if 'K' in x:
        return float(x.replace('K', '')) * 1_000
    if 'M' in x:
        return float(x.replace('M', '')) * 1_000_000
    return float(x)

for col in ['total_followers', 'views', 'likes', 'comments']:
    reels_df[col] = reels_df[col].apply(convert_to_number)

### Standardize dates and the target
Both date columns are text in `m/d/yy` format (for example `1/24/26`), so they are parsed with an explicit format. If any value does not match, the code stops with an error instead of silently creating a missing date. The target is encoded as 0 (baseline) and 1 (high-performing).

In [ ]:
# Both date columns are text in m/d/yy format (for example 1/24/26)
reels_df['date_posted'] = pd.to_datetime(reels_df['date_posted'].astype(str).str.strip(), format='%m/%d/%y')
reels_df['collection_date'] = pd.to_datetime(reels_df['collection_date'].astype(str).str.strip(), format='%m/%d/%y')

reels_df['performance_bucket'] = (
    reels_df['performance_bucket'].astype(str).str.strip().str.lower().map({'baseline': 0, 'high': 1})
)
assert reels_df['performance_bucket'].notna().all(), "Unexpected value in performance_bucket"

Beyond the conflicting duplicate Reels resolved above, no other records were removed. The remaining missing values were expected (see Notebook 1), and no outliers were removed because extreme engagement values are meaningful in the context of social media performance.

## 3.3 Construct data

### Derived features
`engagement_rate` is recalculated from likes, comments, and views, because the raw column is empty. The age of the Reel at collection is calculated in days (weeks would carry exactly the same information).

In [ ]:
# How old the Reel was when performance was measured
reels_df['days_since_posted'] = (reels_df['collection_date'] - reels_df['date_posted']).dt.days

# Outcome measures: normalized so reels from large and small creators can be compared
reels_df['engagement_rate'] = (reels_df['likes'] + reels_df['comments']) / reels_df['views']
reels_df['like_rate'] = reels_df['likes'] / reels_df['views']
reels_df['comment_rate'] = reels_df['comments'] / reels_df['views']
reels_df['views_per_follower'] = reels_df['views'] / reels_df['total_followers']

## 3.4 Explore the cleaned data

This is the second pass of CRISP-DM Data Understanding (the first pass, describing the raw data and checking its quality, is in Notebook 1). It happens here because charts and correlations need the numbers, dates, and standardized labels produced above. `explore_df` is a working copy, so the columns added in this section are not saved with the prepared dataset.

### a) Target variable

In [ ]:
explore_df = reels_df.copy()

target_counts = explore_df['performance_bucket'].value_counts().sort_index()
overall_rate = explore_df['performance_bucket'].mean()

print(target_counts.rename({0: 'baseline (0)', 1: 'high (1)'}))
print(f"\nShare of high-performing reels after cleaning: {overall_rate:.1%}")

The share of high-performing Reels after cleaning is close to the raw data, so the class imbalance noted in Notebook 1 still applies.

### b) Engagement rate and reach by performance group

In [ ]:
explore_df['engagement_rate_pct'] = explore_df['engagement_rate'] * 100

for label, name in [(0, 'Baseline'), (1, 'High')]:
    explore_df.loc[explore_df['performance_bucket'] == label, 'engagement_rate_pct'].hist(
        alpha=0.5, label=name, density=True
    )

plt.title('Engagement Rate by Performance Bucket')
plt.xlabel('Engagement Rate (%)')
plt.ylabel('Density')
plt.legend()
plt.show()

The engagement-rate distributions for baseline and high-performing Reels overlap substantially, so engagement alone does not clearly separate the groups. High-performing Reels have a longer right tail, which means a greater chance of reaching very high engagement values.

In [ ]:
explore_df.boxplot(column='engagement_rate', by='performance_bucket')

plt.title('Engagement Rate by Performance Bucket')
plt.suptitle('')  # removes the automatic subtitle
plt.xlabel('Performance Bucket (0 = Baseline, 1 = High)')
plt.ylabel('Engagement Rate')
plt.show()

Median engagement rates are similar in the two groups, but high-performing Reels show more variability and higher extreme values. Strong performance is associated with engagement outliers more than with consistently higher typical engagement.

In [ ]:
explore_df.boxplot(column='views_per_follower', by='performance_bucket')

plt.title('Views per Follower by Performance Bucket')
plt.suptitle('')
plt.xlabel('Performance Bucket (0 = Baseline, 1 = High)')
plt.ylabel('Views per Follower (Views / Followers)')
plt.yscale('log')
plt.show()

medians = explore_df.groupby('performance_bucket')['views_per_follower'].median()
print(f"Median views per follower: baseline {medians[0]:.2f}, high {medians[1]:.2f} "
      f"(about {medians[1] / medians[0]:.0f}x higher)")

High-performing Reels have a far higher median number of views per follower (see the printed medians), meaning many times greater reach, and the distributions show minimal overlap. The y-axis is logarithmic, so vertical differences are multiplicative rather than additive. Both groups have outliers, but high-performing Reels show more extreme values, indicating occasional viral reach far beyond typical levels.

**Implication:** `views_per_follower` (like views, likes, comments, and engagement rate) describes the *outcome*, so it is not used as a predictor.

### c) Numeric relationships and the age of the Reel

In [ ]:
numeric_cols = [
    'views', 'likes', 'comments', 'engagement_rate',
    'total_followers', 'views_per_follower', 'days_since_posted'
]

# Which numeric variables move with the target?
corr_with_target = (
    explore_df[numeric_cols + ['performance_bucket']]
    .corr()['performance_bucket']
    .drop('performance_bucket')
    .sort_values(ascending=False)
    .round(3)
)
print("Correlation with performance_bucket:")
print(corr_with_target)

plt.figure(figsize=(9, 6))
sns.heatmap(explore_df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Variables')
plt.tight_layout()
plt.show()

In [ ]:
explore_df.boxplot(column='days_since_posted', by='performance_bucket')

plt.title('Days Since Posted by Performance Bucket')
plt.suptitle('')
plt.xlabel('Performance Bucket (0 = Baseline, 1 = High)')
plt.ylabel('Days Between Posting and Data Collection')
plt.show()

Views, likes, and comments are strongly correlated with one another, so they capture overlapping information about engagement volume. A correlation matrix only measures how numeric variables move together, and most predictors in this project are categorical or binary, so the category-level tables and heatmaps below are more informative for the questions of interest.

### Reel age and performance

`days_since_posted` is strongly associated with the target (see the correlation table and the boxplot above). The table below shows how the high-performing rate changes across age groups.

In [ ]:
# Split the reels into three equal-sized age groups
explore_df['age_band'] = pd.qcut(
    explore_df['days_since_posted'], 3, labels=['newest third', 'middle third', 'oldest third']
)

age_summary = explore_df.groupby('age_band', observed=True).agg(
    n_reels=('performance_bucket', 'size'),
    median_age_days=('days_since_posted', 'median'),
    high_rate=('performance_bucket', 'mean'),
)
print(age_summary.round(2))

age_by_group = explore_df.groupby('performance_bucket')['days_since_posted'].median()
print(f"\nMedian age at collection: baseline {age_by_group[0]:.0f} days, high-performing {age_by_group[1]:.0f} days")

The high-performing Reels are much older than the baseline Reels, and the high-performing rate climbs steeply with age. There are two possible explanations, and the data cannot tell them apart: older Reels have had longer to accumulate views, and/or the high-performing Reels were sampled from older posts than the baseline (recent) Reels.

Either way, age is a **confounder**: a variable linked to the outcome that may also be linked to the content features (for example, if some craft categories were posted earlier than others). A confounder can make a content feature look more or less effective than it really is, so section 3.4g checks the category rates within age groups.

### d) Performance rate by content category

For each category, the table shows the number of Reels (`n_reels`) and the share that are high-performing (`high_rate`), compared with the overall rate. Because the rate is calculated within each category, it answers the question of interest: *how likely is a Reel to be high-performing, given its category?*

In [ ]:
def rate_by_category(df, col):
    """Number of reels and high-performing rate for each value of a categorical column."""
    return (
        df.groupby(col)['performance_bucket']
        .agg(n_reels='size', high_rate='mean')
        .sort_values('high_rate', ascending=False)
    )

def plot_rate(summary, title, ylabel):
    fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(summary))))
    labels = [f"{idx} (n={n})" for idx, n in zip(summary.index, summary['n_reels'])]
    ax.barh(labels, summary['high_rate'])
    ax.axvline(overall_rate, linestyle='--', color='gray', label=f"Overall rate ({overall_rate:.1%})")
    ax.invert_yaxis()  # highest rate at the top
    ax.set_title(title)
    ax.set_xlabel('Share of Reels That Are High-Performing')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend()
    plt.tight_layout()
    plt.show()

def describe_rates(summary, min_n=15):
    """Print the highest and lowest rates among categories with at least min_n reels."""
    eligible = summary[summary['n_reels'] >= min_n]
    if len(eligible) < 2:
        return
    top, bottom = eligible.iloc[0], eligible.iloc[-1]
    print(f"Highest rate (n >= {min_n}): {eligible.index[0]} at {top['high_rate']:.0%} ({int(top['n_reels'])} reels)")
    print(f"Lowest rate  (n >= {min_n}): {eligible.index[-1]} at {bottom['high_rate']:.0%} ({int(bottom['n_reels'])} reels)")
    small = summary[summary['n_reels'] < min_n]
    if len(small):
        print(f"Categories with fewer than {min_n} reels (treat as tentative): {', '.join(small.index)}")

In [ ]:
craft_rates = rate_by_category(explore_df, 'craft_category')
print(craft_rates.round(3))
describe_rates(craft_rates)
plot_rate(craft_rates, 'High-Performing Rate by Craft Category', 'Craft Category')

In [ ]:
hook_rates = rate_by_category(explore_df, 'hook_category')
print(hook_rates.round(3))
describe_rates(hook_rates)
plot_rate(hook_rates, 'High-Performing Rate by Hook Category', 'Hook Category')

In [ ]:
structure_rates = rate_by_category(explore_df, 'reel_structure')
print(structure_rates.round(3))
describe_rates(structure_rates)
plot_rate(structure_rates, 'High-Performing Rate by Reel Structure', 'Reel Structure')

Rates based on a small number of Reels can look extreme by chance, so categories with few Reels should be read as tentative. Compare each category's rate with the overall rate (dashed line) and with its sample size.

### e) Delivery features (binary)

In [ ]:
binary_summary = pd.DataFrame({
    col: {
        'high_rate_absent': explore_df.loc[explore_df[col] == 0, 'performance_bucket'].mean(),
        'high_rate_present': explore_df.loc[explore_df[col] == 1, 'performance_bucket'].mean(),
        'n_present': int((explore_df[col] == 1).sum()),
    }
    for col in binary_cols
}).T
binary_summary['difference'] = binary_summary['high_rate_present'] - binary_summary['high_rate_absent']
print(binary_summary.round(3))

ax = binary_summary[['high_rate_absent', 'high_rate_present']].plot(kind='bar', figsize=(8, 5))
ax.set_title('High-Performing Rate by Binary Feature')
ax.set_ylabel('High-Performing Rate')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(['Feature absent', 'Feature present'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The high-performing rate is nearly the same whether or not each delivery feature is present (compare the two bars for each feature and the `difference` column). No single delivery feature separates high-performing Reels from baseline ones on its own, so any effect of these choices would have to show up in combination with other features or be small. Note that `pov_present` is 1 for the vast majority of Reels, so its "absent" group is very small.

### f) Interactions between features

Single-feature views can hide patterns that appear only in combinations (for example, a hook type that works well only with a certain structure). The heatmaps show the high-performing rate for each pair of categories. Cells with fewer than 8 Reels are left blank because their rates are unreliable.

In [ ]:
def interaction_heatmap(df, row_col, col_col, min_n=8):
    counts = pd.crosstab(df[row_col], df[col_col])
    rates = pd.crosstab(df[row_col], df[col_col], values=df['performance_bucket'], aggfunc='mean')
    rates = rates.where(counts >= min_n)  # hide cells with too few reels

    plt.figure(figsize=(max(6, 1.2 * rates.shape[1]), max(4, 0.5 * rates.shape[0])))
    sns.heatmap(rates, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1,
                linewidths=0.5, mask=rates.isna(), cbar_kws={'label': 'High-performing rate'})
    plt.title(f'High-Performing Rate: {row_col} x {col_col}\n(cells with fewer than {min_n} reels are blank)')
    plt.tight_layout()
    plt.show()

interaction_heatmap(explore_df, 'craft_category', 'hook_category')
interaction_heatmap(explore_df, 'craft_category', 'reel_structure')
interaction_heatmap(explore_df, 'hook_category', 'reel_structure')

Cells with enough Reels show how the rate varies across category pairs. Compare each cell with the overall rate and treat cells with few Reels cautiously. The pairs that stood out motivated the **interaction features** built in section 3.5 and tested in the second modeling iteration in Notebook 3.

### g) Are the category differences just a reflection of age?

For the larger categories, the first table shows each category's median age and overall high-performing rate. The second shows the rate within each age group (cells with fewer than 8 Reels are blank), and the third shows how many Reels are in each cell.

In [ ]:
def rate_within_age(df, col, min_category_n=15, min_cell_n=8):
    sizes = df[col].value_counts()
    sub = df[df[col].isin(sizes[sizes >= min_category_n].index)]

    overall = sub.groupby(col).agg(
        n_reels=('performance_bucket', 'size'),
        median_age_days=('days_since_posted', 'median'),
        high_rate=('performance_bucket', 'mean'),
    )
    rates = sub.pivot_table(index=col, columns='age_band', values='performance_bucket',
                            aggfunc='mean', observed=True)
    counts = sub.pivot_table(index=col, columns='age_band', values='performance_bucket',
                             aggfunc='size', observed=True)

    print(f"=== {col} (categories with at least {min_category_n} reels) ===")
    print(overall.round(2), "\n")
    print("High-performing rate within each age group:")
    print(rates.where(counts >= min_cell_n).round(2), "\n")
    print("Reels in each cell:")
    print(counts, "\n")

for col in ['craft_category', 'reel_structure', 'hook_category']:
    rate_within_age(explore_df, col)

Compare each category's overall rate with its rates inside the age groups. Where a category looked strong overall but its advantage mostly disappears within an age group, age explains much of the difference. The cells are small, so these within-group rates are rough, and the main conclusion is that **content effects in this dataset cannot be cleanly separated from Reel age**. This is carried forward as a limitation, and a way to address it (controlling for age or sampling Reels of similar age) is listed as a next step in Notebook 4.

### Summary of the exploration and next steps

| Finding | Next Steps |
|---|---|
| Views, likes, comments, engagement rate, and views per follower describe the outcome | Exclude them from the predictors |
| Reel age is strongly tied to the target and may be tied to some categories | Keep `days_since_posted` out of the predictors, report it as a limitation, and check category rates within age groups (section 3.4g) |
| Several categories contain very few Reels | Drop extremely rare indicator columns before modeling (Notebook 3) |
| Delivery features show almost no difference in rate on their own | Keep them as predictors, but expect any signal to be small or to come from combinations |
| Some category pairs may behave differently from their single-feature rates | Construct interaction features (section 3.5) and test whether they help |

## 3.5 Construct interaction features
The interaction heatmaps in section 3.4f showed that some category pairs behave differently from their single-feature rates. Each interaction feature below equals 1 when a Reel has **both** values in the pair, and 0 otherwise. They are used in the second modeling iteration in Notebook 3.

In [ ]:
# (column A, value A, column B, value B)
# NOTE: reconstructed from the interaction columns used in the evaluation step; verify against
# the original interaction analysis and add or remove pairs as needed.
INTERACTIONS = [
    ('craft_category', 'home_decor',       'hook_category',  'direct_value_delivery'),
    ('craft_category', 'holiday_seasonal', 'reel_structure', 'process'),
    ('craft_category', 'upcycling',        'reel_structure', 'process'),
    ('craft_category', 'plants_garden',    'reel_structure', 'process'),
    ('craft_category', 'home_decor',       'reel_structure', 'showcase'),
    ('hook_category',  'curiosity_gap',        'reel_structure', 'process'),
    ('hook_category',  'direct_value_delivery', 'reel_structure', 'showcase'),
    ('hook_category',  'transformation_tease',  'reel_structure', 'process'),
]

for col_a, val_a, col_b, val_b in INTERACTIONS:
    reels_df[f"int_{val_a}_{val_b}"] = ((reels_df[col_a] == val_a) & (reels_df[col_b] == val_b)).astype(int)

interaction_cols = [c for c in reels_df.columns if c.startswith('int_')]
print("Reels with each interaction:")
print(reels_df[interaction_cols].sum().sort_values(ascending=False))

## 3.6 Format data

Each categorical variable is converted into indicator (0/1) columns. All categories are kept, so every indicator can be read directly (for example `hook_category_curiosity_gap = 1` means the hook is a curiosity gap). Regularization in the logistic regression handles the redundancy between indicator columns.

In [ ]:
reels_df = pd.get_dummies(
    reels_df,
    columns=categorical_cols,
    drop_first=False,
    dtype=int
)

reels_df.head()

### Dimension reduction considerations
- **Correlated numeric variables.** Views, likes, and comments are strongly correlated (section 3.4c). They are outcome measures and are not predictors, so no further variables were removed.
- **PCA.** Principal Component Analysis was considered but not applied. PCA is most useful for reducing a large number of correlated continuous variables, whereas this dataset consists mainly of categorical features converted to binary indicators plus a few numeric variables. PCA would also reduce interpretability, which matters here because the goal is to understand how specific content features relate to performance.

## 3.7 Validate and save the prepared dataset

In [ ]:
print("Shape:", reels_df.shape)
print("Missing values in the prepared data:", int(reels_df.isna().sum().sum()))
print("\nTarget distribution:")
print(reels_df['performance_bucket'].value_counts(normalize=True).round(3))

print("\nColumn groups:")
print("  Binary features:   ", binary_cols)
print("  Category indicators:", len([c for c in reels_df.columns if c.startswith(('craft_category_', 'reel_structure_', 'hook_category_'))]))
print("  Interaction columns:", len(interaction_cols))

In [ ]:
# Export the prepared dataset (read by Notebook 3)
reels_df.to_csv('../Data/cleaned_data.csv', index=False)

### Summary of preparation steps
1. Extracted a unique `reel_id` from the URL, resolved Reels that appeared twice with conflicting labels, and dropped `url` and `hook_text`.
2. Renamed columns, standardized category labels, and encoded the `Y`/`N` variables as 1/0.
3. Converted `K`/`M` text counts to numbers and parsed the dates.
4. Encoded the target as 0 (baseline) and 1 (high-performing).
5. Calculated Reel age, engagement rate, like rate, comment rate, and views per follower.
6. Explored the cleaned data (section 3.4).
7. Built the interaction features and one-hot encoded the categorical variables.
8. Validated the result (no missing values) and saved it for modeling.